# 01 — Data Exploration

**Phase 1 deliverable** from `docs/research_plan.md`.

Goal: ground-truth what QC's free minute data actually looks like for SPY/QQQ/IWM before any indicator work begins. We are checking:

1. **Coverage** — date range, missing bars, half-day handling
2. **Return distribution** — heavy tails, intraday seasonality, day-of-week effects
3. **Volume profile** — intraday shape (U-curve?), absolute scale by symbol, outliers
4. **Resampling sanity** — minute → 15min aggregation correctness

Run this on **QuantConnect Cloud Research** (Free B-MICRO node is enough). Local execution would need `lean research` which is currently blocked by the DinD sidecar memory limit — see CLAUDE.md.

In [ ]:
# region imports
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
# endregion

qb = QuantBook()
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Fetch raw minute data

Start with 1 year first to confirm the pipeline. Extend to 3-5 years once the cells below run cleanly.

In [ ]:
TICKERS = ['SPY', 'QQQ', 'IWM']
END   = datetime(2024, 12, 31)
START = END - timedelta(days=365)

symbols = {t: qb.add_equity(t, Resolution.MINUTE).symbol for t in TICKERS}
history = qb.history(list(symbols.values()), START, END, Resolution.MINUTE)

print(f'rows: {len(history):,}, columns: {list(history.columns)}')
history.head()

## 2. Per-symbol coverage check

Bar count, date range, and unique trading days for each ticker. A regular full-day session is `390` minute bars (9:30–16:00 ET); half-days are `210`.

In [ ]:
for ticker, sym in symbols.items():
    df = history.loc[sym]
    days = df.index.normalize().unique()
    bars_per_day = df.groupby(df.index.date).size()
    print(f'{ticker}: {len(df):>8,} bars | {len(days):>4} sessions | '
          f'bars/day median={int(bars_per_day.median())}, '
          f'half-days={(bars_per_day < 390).sum()}')
    print(f'    range: {df.index.min()} → {df.index.max()}')

## 3. Resample to 15-min and verify

Project spec uses 15-min bars. Confirm the aggregation rule (open=first, high=max, low=min, close=last, volume=sum) and that bar counts match expectations: a 390-minute session → 26 fifteen-minute bars.

In [ ]:
def resample_15min(df: pd.DataFrame) -> pd.DataFrame:
    """OHLCV minute → 15-min. Index must already be DatetimeIndex."""
    return df.resample('15min').agg({
        'open':   'first',
        'high':   'max',
        'low':    'min',
        'close':  'last',
        'volume': 'sum',
    }).dropna(subset=['open'])

spy_1m  = history.loc[symbols['SPY']]
spy_15m = resample_15min(spy_1m)

print(f'1m bars : {len(spy_1m):>8,}')
print(f'15m bars: {len(spy_15m):>8,}  (expect ≈ 1m / 15)')
print()
spy_15m.head(4)

## 4. Return distribution

Per-bar log returns. We expect:
- Centered near zero, slightly negative skew
- Heavy tails (kurtosis ≫ 3)
- Significant intraday seasonality (open/close volatility ≫ mid-day)

If our IC analysis later assumes normality, this is where we confirm the assumption is wrong and pick robust statistics (Spearman, not Pearson).

In [ ]:
spy_15m['ret'] = np.log(spy_15m['close'] / spy_15m['close'].shift(1))
r = spy_15m['ret'].dropna()

print('SPY 15-min log returns')
print(f'  mean    : {r.mean():+.6f}')
print(f'  std     : {r.std():.6f}')
print(f'  skew    : {r.skew():+.4f}')
print(f'  kurtosis: {r.kurt():+.4f}  (excess; normal=0)')
print(f'  min/max : {r.min():+.4f} / {r.max():+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(r, bins=80)
axes[0].set_title('SPY 15m return histogram')
axes[0].axvline(0, color='red', alpha=0.5)
axes[1].plot(r.rolling(96).std() * np.sqrt(96 * 252))   # 96 bars/day → annualized vol
axes[1].set_title('SPY rolling 1-day annualized vol')
plt.tight_layout()

## 5. Intraday volume profile

Average volume per 15-min slot. The textbook U-curve (heavy at open and close, lighter mid-day) is what we'll filter against in Combo 1 ("current bar vs same-time-of-day average"). Confirm the shape before relying on it.

In [ ]:
spy_15m['minute_of_day'] = spy_15m.index.hour * 60 + spy_15m.index.minute
by_slot = spy_15m.groupby('minute_of_day')['volume'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_slot.index, by_slot.values, width=12)
ax.set_xlabel('Minute of day (ET, market hours)')
ax.set_ylabel('Mean SPY volume per 15-min bar')
ax.set_title('Intraday volume profile (15-min bars)')
plt.tight_layout()

## 6. Sanity check: ICAnalyzer on a useless indicator

Quick test that the project's `analysis/ic_calculator` works end-to-end. We use a pure-noise indicator — IC should be near zero and not significant. This is the contract that future cells will trust when they evaluate real indicators.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.dirname(os.getcwd())   # ../ relative to research/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from analysis.ic_calculator import compute_ic, quantile_test, ic_summary

np.random.seed(0)
noise = pd.Series(np.random.randn(len(spy_15m)), index=spy_15m.index)
per_bar_ret = spy_15m['close'].pct_change()

print('noise vs SPY forward returns:')
print(ic_summary(noise, per_bar_ret, periods=(1, 4, 24)).to_string())
print()
print('quantile_test (means should look unstructured):')
print(quantile_test(noise, per_bar_ret, n_quantiles=5).to_string())

## Next steps

Once this notebook runs end-to-end on the cloud research node:

- Move to `02_indicator_analysis.ipynb`: implement VWAP, OBV, MFI, A/D, CMF; run `compute_ic` + `quantile_test` against each; drop any with `|IC| < 0.02`.
- Extend the data window from 1 year → 3-5 years.
- Save the cleaned 15-min DataFrames to the QC Object Store so we don't re-fetch every notebook restart.